# Flow matching on two moons (optional)
**LLMs, Animated · Lesson G1.5**

You'll train a flow-matching model (Lipman et al., 2022; Liu et al., 2022) on the same two-moons toy as the lesson, then sample it with 1, 4, 16 and 64 Euler steps and measure the difference:

1. the straight-line interpolation and its velocity target,
2. the flow-matching loss,
3. an Euler sampler,
4. a quality metric to compare step counts.

The experiment at the end tries **reflow**. Each exercise has a `# TODO` cell followed by a **check** cell with `assert`s. Everything runs on a CPU in well under a minute; solutions are at the bottom.

Convention (as in the lesson): $x_0$ is **noise**, $x_1$ is **data**, $x_t = (1-t)\,x_0 + t\,x_1$ for $t \in [0, 1]$.

In [ ]:
import math, time
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)
torch.set_num_threads(2)

def make_moons(n, noise=0.08, seed=None):
    g = torch.Generator().manual_seed(seed) if seed is not None else None
    th = torch.rand(n, generator=g) * math.pi
    up = torch.rand(n, generator=g) < 0.5
    x = torch.where(up, torch.cos(th), 1 - torch.cos(th))
    y = torch.where(up, torch.sin(th), 0.5 - torch.sin(th))
    pts = (torch.stack([x, y], 1) - torch.tensor([0.5, 0.25])) * 1.3
    return pts + noise * torch.randn(n, 2, generator=g)

ref = make_moons(5000, seed=1)   # reference set for measuring sample quality

## 1 · Interpolation and target
Write `interpolate(x0, x1, t)` returning $x_t = (1-t)\,x_0 + t\,x_1$ and `target_velocity(x0, x1)` returning $x_1 - x_0$. Points have shape `(B, 2)`; `t` has shape `(B,)` (use `t[:, None]` to broadcast).

In [ ]:
def interpolate(x0, x1, t):
    # TODO
    raise NotImplementedError("Exercise 1")

def target_velocity(x0, x1):
    # TODO
    raise NotImplementedError("Exercise 1")

**Check 1.** The lesson's worked example: $x_0 = (0.4, -1.2)$, $x_1 = (1.0, 0.5)$, $t = 0.25$.

In [ ]:
x0 = torch.tensor([[0.4, -1.2]]); x1 = torch.tensor([[1.0, 0.5]]); t = torch.tensor([0.25])
xt = interpolate(x0, x1, t); v = target_velocity(x0, x1)
assert torch.allclose(xt, torch.tensor([[0.55, -0.775]])), xt
assert torch.allclose(v, torch.tensor([[0.6, 1.7]])), v
assert torch.allclose(xt + 0.75 * v, x1), "one exact Euler step along a straight line lands on x1"
print("✓ interpolation OK")

The velocity network is a small MLP that sees the point and a few Fourier features of $t$ (provided):

In [ ]:
class VelocityNet(nn.Module):
    def __init__(self, hidden=128, nfreq=8):
        super().__init__()
        self.register_buffer("freqs", torch.arange(1, nfreq + 1).float() * math.pi)
        self.net = nn.Sequential(
            nn.Linear(2 + 2 * nfreq, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2))
    def forward(self, x, t):
        a = t[:, None] * self.freqs[None, :]
        return self.net(torch.cat([x, torch.sin(a), torch.cos(a)], dim=1))

## 2 · The flow-matching loss
Write `fm_loss(model, x0, x1, t)`: interpolate, predict `model(x_t, t)`, and return the mean squared error to the target velocity.

In [ ]:
def fm_loss(model, x0, x1, t):
    # TODO
    raise NotImplementedError("Exercise 2")

In [ ]:
x0 = torch.randn(32, 2); x1 = make_moons(32, seed=3); t = torch.rand(32)
oracle = lambda x, tt: x1 - x0          # knows each pair's true velocity
zero = lambda x, tt: torch.zeros_like(x)
assert fm_loss(oracle, x0, x1, t).item() < 1e-10
assert torch.allclose(fm_loss(zero, x0, x1, t), ((x1 - x0) ** 2).mean())
print("✓ loss OK")

Train (about 10–20 seconds). Each step pairs fresh noise with fresh data **at random**, exactly as in the lesson.

In [ ]:
def train(pairs=None, steps=3000, seed=0):
    torch.manual_seed(seed)
    model = VelocityNet(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for step in range(steps):
        if pairs is None:
            x1 = make_moons(512); x0 = torch.randn_like(x1)
        else:  # reuse given (noise, data) pairs; used by the reflow experiment
            idx = torch.randint(0, len(pairs[0]), (512,)); x0, x1 = pairs[0][idx], pairs[1][idx]
        loss = fm_loss(model, x0, x1, torch.rand(512))
        opt.zero_grad(); loss.backward(); opt.step()
    return model

t0 = time.time(); model = train()
print(f"trained in {time.time() - t0:.1f}s")

## 3 · Euler sampling
Write `euler_sample(model, z, n_steps)`: start at `x = z` (noise), and for `i = 0 … n_steps-1` set $t = i/N$ and take $x \leftarrow x + \frac{1}{N}\, v_\theta(x, t)$. Use `torch.no_grad()` and return the final `x`.

In [ ]:
def euler_sample(model, z, n_steps):
    # TODO
    raise NotImplementedError("Exercise 3")

**Check 3.** With one step every sample should land near the data **mean** (the lesson's "blurry average"); with 64 steps they spread over the moons.

In [ ]:
z = torch.randn(1000, 2, generator=torch.Generator().manual_seed(3))
one = euler_sample(model, z, 1); many = euler_sample(model, z, 64)
print("1 step: spread", one.std(0).tolist(), " 64 steps: spread", many.std(0).tolist())
assert one.std(0).max() < 0.3, "one step should collapse samples toward the mean"
assert many.std(0)[0] > 0.8
print("✓ sampler OK")

## 4 · Compare step counts
Write `mean_nn_dist(x)`: the average, over samples, of the distance to the **nearest** point in `ref` (hint: `torch.cdist(x, ref).min(dim=1).values`). Lower means samples sit closer to the data. Then the cell below compares 1, 4, 16 and 64 steps.

In [ ]:
def mean_nn_dist(x):
    # TODO
    raise NotImplementedError("Exercise 4")

In [ ]:
results = {}
fig, axs = plt.subplots(1, 4, figsize=(13, 3))
for ax, n in zip(axs, [1, 4, 16, 64]):
    s = euler_sample(model, z, n); results[n] = mean_nn_dist(s)
    ax.scatter(s[:, 0], s[:, 1], s=2, c="C1"); ax.set_title(f"{n} step(s): dist {results[n]:.3f}")
    ax.set_xlim(-3, 3); ax.set_ylim(-2.2, 2.2)
plt.show()
print(results)
assert results[1] > results[4] > results[64] - 0.01, "more steps should not be worse"
assert results[64] < 0.1
print("✓ comparison OK")

## Experiment: reflow
The plain model needs several steps because random pairs make the training lines cross. **Reflow** (Liu et al., 2022) re-pairs each noise point with where the current model takes it, then retrains on those pairs. Run the cell (about 5–10 seconds) and compare 1-step samples before and after.

Things to try: reflow with fewer generated pairs; a second round of reflow; or draw the straight lines from `zz[:20]` to `xx[:20]` and compare with lines between random pairs.

In [ ]:
t0 = time.time()
zz = torch.randn(5000, 2)
xx = euler_sample(model, zz, 64)            # where the current flow takes each noise point
model2 = train(pairs=(zz, xx), steps=1500, seed=1)
print(f"reflow trained in {time.time() - t0:.1f}s")
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, m, name in [(axs[0], model, "before reflow"), (axs[1], model2, "after reflow")]:
    s = euler_sample(m, z, 1)
    ax.scatter(s[:, 0], s[:, 1], s=2, c="C1"); ax.set_title(f"{name}, 1 step: dist {mean_nn_dist(s):.3f}")
    ax.set_xlim(-3, 3); ax.set_ylim(-2.2, 2.2)
plt.show()

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def interpolate(x0, x1, t):
    return (1 - t)[:, None] * x0 + t[:, None] * x1

def target_velocity(x0, x1):
    return x1 - x0

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def fm_loss(model, x0, x1, t):
    xt = interpolate(x0, x1, t)
    return ((model(xt, t) - target_velocity(x0, x1)) ** 2).mean()

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def euler_sample(model, z, n_steps):
    x = z.clone()
    with torch.no_grad():
        for i in range(n_steps):
            t = torch.full((len(x),), i / n_steps)
            x = x + model(x, t) / n_steps
    return x

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def mean_nn_dist(x):
    return torch.cdist(x, ref).min(dim=1).values.mean().item()